In [1]:
# ============================================================
#  CLASE: WORKING WITH LLMs - API DE OPENAI
#  Rol: Analista de Datos e IA
#  Versión didáctica para alumnos principiantes
# ============================================================

from openai import OpenAI
import getpass
import json

# ------------------------------------------------------------
#  CONFIGURACIÓN
# ------------------------------------------------------------
api_key = getpass.getpass("🔑 Ingresa tu API KEY de OpenAI: ")
client = OpenAI(api_key=api_key)
print(" Listo!\n")

# Definimos UN solo rol que se reutiliza en todos los ejemplos
ROL = "Eres un analista de datos amable que explica resultados de forma clara y simple."

# Datos de ejemplo: ventas mensuales de una pequeña tienda
DATOS = """
Ventas del último trimestre:
- Enero:   $12.000
- Febrero: $9.500
- Marzo:   $15.200
"""


🔑 Ingresa tu API KEY de OpenAI:  ········


 Listo!



In [4]:

# ============================================================
#  BLOQUE 1: PRIMERA LLAMADA A LA API
# ------------------------------------------------------------
#  Conceptos: model, system prompt, user prompt, temperature
# ============================================================
#print("=" * 50)
#print("  BLOQUE 1: Tu primera llamada al LLM")
#print("=" * 50)

respuesta = client.chat.completions.create(
    model="gpt-4o-mini",                 # Modelo: rápido y económico
    messages=[
        {"role": "system", "content": ROL},        # Define cómo se comporta
        {"role": "user", "content": f"Analiza estas ventas:\n{DATOS}"}
    ],
    temperature=0.3,                     # 0 = preciso | 1 = creativo
    max_tokens=800,                      # Largo máximo de la respuesta
)

#print(respuesta.choices[0].message.content)
#print(f"\n📊 Tokens usados: {respuesta.usage.total_tokens}")




In [5]:
# ============================================================
#  BLOQUE 2: EFECTO DE TEMPERATURE
# ------------------------------------------------------------
#  Mismo prompt, distinta temperature → distinto estilo
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 2: Cómo cambia la respuesta con temperature")
print("=" * 50)

pregunta = "Sugiere un nombre creativo para un dashboard de ventas."

for temp in [0, 1.5]:
    print(f"\n--- temperature = {temp} ---")
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ROL},
            {"role": "user", "content": pregunta}
        ],
        temperature=temp,
        max_tokens=800,
    )
    print(r.choices[0].message.content)

print("\n💡 temperature baja = consistente | temperature alta = creativo")




  BLOQUE 2: Cómo cambia la respuesta con temperature

--- temperature = 0 ---
¡Claro! Aquí tienes algunas sugerencias creativas para un nombre de dashboard de ventas:

1. **"Ventas en Vista"**
2. **"Rumbo a las Ventas"**
3. **"El Radar de Ventas"**
4. **"Ventas al Instante"**
5. **"Métricas de Éxito"**
6. **"El Pulso de las Ventas"**
7. **"Ventas en Tiempo Real"**
8. **"Navegador de Negocios"**
9. **"Tablero de Triunfos"**
10. **"Visión de Ventas"**

Espero que alguna de estas opciones te inspire. ¡Si necesitas más ideas, no dudes en pedirlas!

--- temperature = 1.5 ---
Claro, aquí tienes algunas sugerencias creativas para un dashboard de ventas:

1. **"Insights de Éxito"**
2. **"Mapa de Ventas"**
3. **"Vista Aérea de Ventas"**
4. **"Rendimiento Sónico"**
5. **"Peldaños de Prosperidad"**
6. **"ConexIÓN de Ventas"**
7. **"Sales Sparks: Destellos de Éxito"**
8. **"Tendencias Triunfadoras"**

Elige el que más te resuene o te inspire. ¡En cualquier caso, el objetivo es que642 hiloto gener

In [6]:

# ============================================================
#  BLOQUE 3: SALIDA EN JSON (para usar en un sistema)
# ------------------------------------------------------------
#  Cuando queremos integrar el LLM con código o una BD
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 3: Respuesta en formato JSON")
print("=" * 50)

r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": ROL + " Devuelve SIEMPRE un JSON con: tendencia, mes_mayor, mes_menor, recomendacion."
        },
        {"role": "user", "content": DATOS}
    ],
    temperature=0,
    response_format={"type": "json_object"},
    max_tokens=200,
)

datos = json.loads(r.choices[0].message.content)
print(json.dumps(datos, indent=2, ensure_ascii=False))
print("\n💡 Este JSON ya se puede guardar en una base de datos o mostrar en un dashboard.")






  BLOQUE 3: Respuesta en formato JSON
{
  "tendencia": "Aumento en las ventas",
  "mes_mayor": "Marzo",
  "mes_menor": "Febrero",
  "recomendacion": "Mantener las estrategias que impulsaron las ventas en marzo y analizar las causas de la baja en febrero para mejorar en el futuro."
}

💡 Este JSON ya se puede guardar en una base de datos o mostrar en un dashboard.


In [7]:

# ============================================================
#  BLOQUE 4: CONVERSACIÓN CON MEMORIA (multi-turno)
# ------------------------------------------------------------
#  El LLM no recuerda nada → tenemos que enviarle el historial
# ============================================================
print("\n" + "=" * 50)
print("  BLOQUE 4: Conversación con contexto")
print("=" * 50)
print("Escribe tus preguntas. Escribe 'salir' para terminar.\n")

historial = [
    {"role": "system", "content": ROL},
    {"role": "user", "content": f"Te paso estos datos:\n{DATOS}"}
]

# Primera respuesta del bot para arrancar
r = client.chat.completions.create(
    model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=150
)
historial.append({"role": "assistant", "content": r.choices[0].message.content})
print(f"🤖 Analista: {r.choices[0].message.content}\n")

# Loop de conversación
while True:
    pregunta = input("👤 Tú: ").strip()
    if pregunta.lower() == "salir":
        break
    historial.append({"role": "user", "content": pregunta})

    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=historial, temperature=0.3, max_tokens=200
    )
    respuesta = r.choices[0].message.content
    historial.append({"role": "assistant", "content": respuesta})
    print(f"🤖 Analista: {respuesta}\n")

print(f"\n Cada turno reenvía TODO el historial. Por eso decimos que el LLM es 'sin memoria'.")
print("\n ¡Fin de la clase! Ya viste los 4 conceptos clave para trabajar con LLMs.")


  BLOQUE 4: Conversación con contexto
Escribe tus preguntas. Escribe 'salir' para terminar.

🤖 Analista: ¡Claro! Vamos a analizar esos datos de ventas del último trimestre.

1. **Total de Ventas**: Primero, sumemos las ventas de los tres meses:
   - Enero: $12,000
   - Febrero: $9,500
   - Marzo: $15,200

   Total = $12,000 + $9,500 + $15,200 = **$36,700**

2. **Promedio de Ventas**: Ahora, calculemos el promedio de ventas por mes:
   - Promedio = Total de Ventas / Número de Meses
   - Promedio = $36,700 / 3 ≈ **$12,233.33**

3. **Mes con



👤 Tú:  mayores ventas


🤖 Analista: El mes con mayores ventas fue **marzo**, con un total de **$15,200**. 

Aquí tienes un resumen:

- **Enero**: $12,000
- **Febrero**: $9,500
- **Marzo**: $15,200 (mes con mayores ventas)

Si necesitas más análisis o información, ¡no dudes en preguntar!



👤 Tú:  a que se debe el gran aumento en marzo? 


🤖 Analista: El aumento en las ventas de marzo en comparación con enero y febrero puede deberse a varios factores. Aquí hay algunas posibles explicaciones:

1. **Estacionalidad**: Marzo puede ser un mes en el que tradicionalmente se realizan más compras, como el inicio de la primavera o eventos específicos (por ejemplo, promociones de primavera, días festivos, etc.).

2. **Promociones o Descuentos**: Si se llevaron a cabo campañas de marketing, descuentos o promociones especiales en marzo, esto podría haber incentivado a más clientes a realizar compras.

3. **Lanzamiento de Nuevos Productos**: Si se introdujeron nuevos productos o servicios en marzo, esto podría haber atraído a más clientes y aumentado las ventas.

4. **Recuperación Económica**: Si hubo una mejora en la situación económica o en la confianza del consumidor en marzo, esto podría haber llevado a un aumento en el gasto.

5. **Eventos Especiales**: Eventos como ferias



👤 Tú:  0


🤖 Analista: Parece que tu mensaje se cortó o no se envió completamente. Si tienes alguna pregunta específica o necesitas más información sobre el aumento en las ventas de marzo, no dudes en decírmelo. Estoy aquí para ayudarte.



👤 Tú:  Esc


🤖 Analista: Parece que tu mensaje se ha cortado nuevamente. Si necesitas ayuda con algo específico o si quieres continuar con el análisis de las ventas, por favor, proporciona más detalles. Estoy aquí para ayudarte.



KeyboardInterrupt: Interrupted by user